# Visualizações de teste
### Fundação Pão dos Pobres

Este notebook é um laboratório para testar visualizações antes de levar as melhores para o **Streamlit**.

## 1. Importação das bibliotecas

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
import plotly.graph_objects as go

## 2. Leitura do dataset tratado

Este código procura o arquivo `lem_consolidado_2021_2025.csv` em algumas pastas comuns do projeto. Se o arquivo estiver em outro lugar, ajuste a variável `possiveis_caminhos`.

In [2]:
possiveis_caminhos = [
    Path("dados_tratados/lem_consolidado_2021_2025.csv"),
    Path("lem_consolidado_2021_2025.csv"),
    Path("../dados_tratados/lem_consolidado_2021_2025.csv"),
    Path("../lem_consolidado_2021_2025.csv"),
]

caminho_dataset = None
for caminho in possiveis_caminhos:
    if caminho.exists():
        caminho_dataset = caminho
        break

if caminho_dataset is None:
    raise FileNotFoundError(
        "Arquivo lem_consolidado_2021_2025.csv não encontrado. "
        "Coloque o CSV em dados_tratados/ ou ajuste a lista possiveis_caminhos."
    )

print(f"Dataset encontrado em: {caminho_dataset}")
df = pd.read_csv(caminho_dataset)
df.head()

Dataset encontrado em: dados_tratados/lem_consolidado_2021_2025.csv


,ano,mes_numero,mes,data,categoria,subcategoria,indicador,indicador_completo,valor
0,2021,1,JAN,2021-01-01,EDUCAÇÃO,Ensino EJA,aguardando vaga,Ensino EJA - aguardando vaga,0.0
1,2021,1,JAN,2021-01-01,EDUCAÇÃO,Ensino EJA,matriculados,Ensino EJA - matriculados,0.0
2,2021,1,JAN,2021-01-01,EDUCAÇÃO,Ensino infantil,aguardando vaga,Ensino infantil - aguardando vaga,3.0
3,2021,1,JAN,2021-01-01,EDUCAÇÃO,Ensino infantil,matriculados,Ensino infantil - matriculados,2.0
4,2021,1,JAN,2021-01-01,EDUCAÇÃO,Ensino regular,aguardando vaga,Ensino regular - aguardando vaga,3.0


## 3. Ajuste dos tipos e preparação inicial

In [3]:
df["data"] = pd.to_datetime(df["data"], errors="coerce")
df["valor"] = pd.to_numeric(df["valor"], errors="coerce").fillna(0)

# Garante ordenação temporal correta
df = df.sort_values(["data", "categoria", "indicador_completo"]).reset_index(drop=True)

print(df.info())
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 1862 entries, 0 to 1861
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   ano                 1862 non-null   int64         
 1   mes_numero          1862 non-null   int64         
 2   mes                 1862 non-null   str           
 3   data                1862 non-null   datetime64[us]
 4   categoria           850 non-null    str           
 5   subcategoria        436 non-null    str           
 6   indicador           1862 non-null   str           
 7   indicador_completo  1862 non-null   str           
 8   valor               1862 non-null   float64       
dtypes: datetime64[us](1), float64(1), int64(2), str(5)
memory usage: 244.4 KB
None


,ano,mes_numero,mes,data,categoria,subcategoria,indicador,indicador_completo,valor
0,2021,1,JAN,2021-01-01,EDUCAÇÃO,Ensino EJA,aguardando vaga,Ensino EJA - aguardando vaga,0.0
1,2021,1,JAN,2021-01-01,EDUCAÇÃO,Ensino EJA,matriculados,Ensino EJA - matriculados,0.0
2,2021,1,JAN,2021-01-01,EDUCAÇÃO,Ensino infantil,aguardando vaga,Ensino infantil - aguardando vaga,3.0
3,2021,1,JAN,2021-01-01,EDUCAÇÃO,Ensino infantil,matriculados,Ensino infantil - matriculados,2.0
4,2021,1,JAN,2021-01-01,EDUCAÇÃO,Ensino regular,aguardando vaga,Ensino regular - aguardando vaga,3.0


## 4. Filtros de teste

No notebook, os filtros ficam como variáveis. Depois, no Streamlit, essas variáveis viram `st.multiselect`, `st.selectbox` ou `st.slider`.

In [4]:
anos_disponiveis = sorted(df["ano"].dropna().unique())
categorias_disponiveis = sorted(df["categoria"].dropna().unique())
indicadores_disponiveis = sorted(df["indicador_completo"].dropna().unique())

# Ajuste aqui para testar recortes diferentes
anos_selecionados = anos_disponiveis
categorias_selecionadas = categorias_disponiveis

# Para não gerar gráficos gigantes, começa com os 8 indicadores de maior volume
top_indicadores = (
    df.groupby("indicador_completo", as_index=False)["valor"].sum()
    .sort_values("valor", ascending=False)
    .head(8)["indicador_completo"]
    .tolist()
)

indicadores_selecionados = top_indicadores

df_filtrado = df[
    (df["ano"].isin(anos_selecionados)) &
    (df["categoria"].isin(categorias_selecionadas)) &
    (df["indicador_completo"].isin(indicadores_selecionados))
].copy()

df_filtrado.shape, indicadores_selecionados

((116, 9),
 ['Atendimentos individual',
  'Atendimento familiar',
  'Saúde mental',
  'Interface com educação',
  'Efetivos na casa',
  'Interface com rede socioassistencial',
  'Saúde clínica',
  'Interface com saúde'])

# 5. Pergunta de Negócio e Visualização

## 5.1 Evolução da demanda e períodos de sobrecarga

### V1 — Small multiples com linhas por indicador

**Objetivo:** comparar a evolução mensal de diferentes indicadores sem sobrepor muitas linhas no mesmo gráfico.

**Quando usar no Streamlit:** quando o objetivo for mostrar tendências, picos e quedas de cada indicador separadamente.

In [5]:
df_small = (
    df_filtrado
    .groupby(["data", "ano", "mes", "mes_numero", "indicador_completo"], as_index=False)
    .agg(valor=("valor", "sum"))
    .sort_values("data")
)

fig = px.line(
    df_small,
    x="data",
    y="valor",
    facet_col="indicador_completo",
    facet_col_wrap=2,
    markers=True,
    title="Evolução mensal dos atendimentos técnicos por indicador",
    labels={
        "data": "Período",
        "valor": "Quantidade de atendimentos",
        "indicador_completo": "Indicador"
    },
    hover_data={
        "ano": True,
        "mes": True,
        "valor": True,
        "indicador_completo": True,
        "data": "|%m/%Y"
    }
)

fig.update_traces(
    hovertemplate=(
        "<b>Indicador:</b> %{customdata[3]}<br>" +
        "<b>Mês:</b> %{customdata[1]}<br>" +
        "<b>Ano:</b> %{customdata[0]}<br>" +
        "<b>Atendimentos:</b> %{y}<extra></extra>"
    )
)

fig.update_layout(height=900, showlegend=False)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

### V2 — Linha única do total mensal

**Objetivo:** mostrar a evolução geral da demanda, somando todos os indicadores selecionados.

**Quando usar no Streamlit:** quando a Fundação quiser uma visão executiva rápida da sobrecarga total mês a mês.

In [6]:
df_total_mensal = (
    df_filtrado
    .groupby(["data", "ano", "mes", "mes_numero"], as_index=False)
    .agg(total_atendimentos=("valor", "sum"))
    .sort_values("data")
)

fig = px.line(
    df_total_mensal,
    x="data",
    y="total_atendimentos",
    markers=True,
    title="Evolução mensal da demanda total de atendimentos técnicos",
    labels={"data": "Período", "total_atendimentos": "Total de atendimentos"},
    hover_data={"ano": True, "mes": True, "data": "|%m/%Y"}
)

fig.update_traces(
    hovertemplate=(
        "<b>Mês:</b> %{customdata[1]}<br>" +
        "<b>Ano:</b> %{customdata[0]}<br>" +
        "<b>Total de atendimentos:</b> %{y}<extra></extra>"
    )
)

fig.show()

### V3 — Linha com média móvel

**Objetivo:** suavizar oscilações mensais e identificar a tendência geral da demanda.

**Quando usar no Streamlit:** se os dados tiverem muitos altos e baixos e vocês quiserem destacar a tendência.

In [7]:
df_total_mensal["media_movel_3m"] = df_total_mensal["total_atendimentos"].rolling(window=3, min_periods=1).mean()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_total_mensal["data"],
    y=df_total_mensal["total_atendimentos"],
    mode="lines+markers",
    name="Total mensal",
    customdata=df_total_mensal[["mes", "ano"]],
    hovertemplate="<b>Mês:</b> %{customdata[0]}<br><b>Ano:</b> %{customdata[1]}<br><b>Total:</b> %{y}<extra></extra>"
))

fig.add_trace(go.Scatter(
    x=df_total_mensal["data"],
    y=df_total_mensal["media_movel_3m"],
    mode="lines",
    name="Média móvel de 3 meses",
    hovertemplate="<b>Média móvel:</b> %{y:.1f}<extra></extra>"
))

fig.update_layout(
    title="Demanda total mensal com média móvel de 3 meses",
    xaxis_title="Período",
    yaxis_title="Total de atendimentos"
)

fig.show()

### V4 — Heatmap de sobrecarga por mês e ano

**Objetivo:** identificar rapidamente quais meses e anos concentraram maior volume de atendimentos.

**Quando usar no Streamlit:** ótima opção para responder diretamente “quais períodos indicam maior sobrecarga”.

In [8]:
df_heatmap = df_total_mensal.copy()

ordem_meses = (
    df[["mes_numero", "mes"]]
    .drop_duplicates()
    .sort_values("mes_numero")
)
meses_ordenados = ordem_meses["mes"].tolist()

fig = px.density_heatmap(
    df_heatmap,
    x="mes",
    y="ano",
    z="total_atendimentos",
    category_orders={"mes": meses_ordenados},
    text_auto=True,
    title="Mapa de calor da demanda total por mês e ano",
    labels={"mes": "Mês", "ano": "Ano", "total_atendimentos": "Total de atendimentos"}
)

fig.update_layout(height=500)
fig.show()

## 5.2 Evolução dos desdobramentos técnicos

Filtro específico para localizar os registros da segunda pergunta. Na base final, os desdobramentos técnicos aparecem como registros sem categoria preenchida; por isso criamos uma categoria ajustada apenas para as visualizações.

In [9]:
# Na base tratada, os registros de DESDOBRAMENTOS TÉCNICOS ficaram com categoria vazia/None.
# Por isso o filtro por texto "desdobramento" não encontrava nada.
# Aqui criamos uma categoria ajustada apenas para as visualizações.

df["categoria_ajustada"] = df["categoria"].fillna("DESDOBRAMENTOS TÉCNICOS")

df_desdobramentos = df[
    df["categoria_ajustada"] == "DESDOBRAMENTOS TÉCNICOS"
].copy()

print("Quantidade de registros encontrados:", len(df_desdobramentos))

display(
    df_desdobramentos[
        ["ano", "mes", "categoria_ajustada", "subcategoria", "indicador", "indicador_completo", "valor"]
    ].head()
)

print("Tipos de desdobramentos técnicos encontrados:")
display(
    sorted(df_desdobramentos["indicador_completo"].dropna().unique())
)


Quantidade de registros encontrados: 1012


,ano,mes,categoria_ajustada,subcategoria,indicador,indicador_completo,valor
16,2021,JAN,DESDOBRAMENTOS TÉCNICOS,NaN,Apadrinhamento afetivo,Apadrinhamento afetivo,0.0
17,2021,JAN,DESDOBRAMENTOS TÉCNICOS,NaN,Atendimento familiar,Atendimento familiar,31.0
18,2021,JAN,DESDOBRAMENTOS TÉCNICOS,NaN,Atendimentos individual,Atendimentos individual,40.0
19,2021,JAN,DESDOBRAMENTOS TÉCNICOS,NaN,Colocação em família substituta,Colocação em família substituta,0.0
20,2021,JAN,DESDOBRAMENTOS TÉCNICOS,NaN,Desligamentos,Desligamentos,0.0


Tipos de desdobramentos técnicos encontrados:


['Apadrinhamento afetivo',
 'Atendimento familiar',
 'Atendimentos individual',
 'Colocação em família substituta',
 'Desligamentos',
 'Documentação civil (RG / CPF/ CTPS / Certidão de nascimento)',
 'Efetivos na casa',
 'Evasão',
 'Interface com educação',
 'Interface com judiciário',
 'Interface com rede socioassistencial',
 'Interface com saúde',
 'Novos ingressos',
 'PIAs / Relatórios',
 'Reunião de equipe (técnica, casa, coordenação, gerente)',
 'Transferências',
 'Visitas Domiciliares']

#### Filtro de anos para teste

No Streamlit, esse filtro pode virar um seletor de anos por checkbox ou multiselect.  
No notebook, ele fica como uma lista simples para facilitar os testes.

In [10]:
# Filtros para testar no notebook.
# Depois, no Streamlit, isso vira um seletor de anos por checkbox/multiselect.

anos_disponiveis_q2 = sorted(df_desdobramentos["ano"].dropna().unique())
anos_selecionados_q2 = anos_disponiveis_q2

df_desdobramentos_filtrado = df_desdobramentos[
    df_desdobramentos["ano"].isin(anos_selecionados_q2)
].copy()

print("Anos disponíveis:", anos_disponiveis_q2)
print("Anos selecionados:", anos_selecionados_q2)
print("Quantidade após filtro:", len(df_desdobramentos_filtrado))


Anos disponíveis: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Anos selecionados: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Quantidade após filtro: 1012


### V1 — Gráfico de linha por tipo de desdobramento técnico

**Objetivo:** acompanhar a evolução temporal dos tipos de desdobramentos técnicos.

**Quando usar no Streamlit:**  
Esta é a visualização principal para a segunda pergunta. Ela mostra aumentos, quedas e oscilações ao longo do tempo e permite comparar quais tipos de desdobramentos ganharam ou perderam relevância.

In [11]:
df_linha_desdobramentos = (
    df_desdobramentos_filtrado
    .groupby(["data", "ano", "mes", "mes_numero", "indicador_completo"], as_index=False)
    .agg(valor=("valor", "sum"))
    .sort_values("data")
)

# Para evitar um gráfico poluído, pega os principais tipos de desdobramento pelo volume total.
top_tipos_desdobramento = (
    df_linha_desdobramentos
    .groupby("indicador_completo", as_index=False)
    .agg(total=("valor", "sum"))
    .sort_values("total", ascending=False)
    .head(8)["indicador_completo"]
)

df_linha_top = df_linha_desdobramentos[
    df_linha_desdobramentos["indicador_completo"].isin(top_tipos_desdobramento)
]

fig = px.line(
    df_linha_top,
    x="data",
    y="valor",
    color="indicador_completo",
    markers=True,
    title="Evolução dos principais tipos de desdobramentos técnicos ao longo do tempo",
    labels={
        "data": "Período",
        "valor": "Quantidade",
        "indicador_completo": "Tipo de desdobramento técnico"
    },
    hover_data={
        "ano": True,
        "mes": True,
        "valor": True,
        "indicador_completo": True
    }
)

fig.update_traces(
    hovertemplate=
    "<b>Tipo:</b> %{customdata[3]}<br>" +
    "<b>Mês:</b> %{customdata[1]}<br>" +
    "<b>Ano:</b> %{customdata[0]}<br>" +
    "<b>Quantidade:</b> %{y}<extra></extra>"
)

fig.update_layout(
    height=650,
    legend_title_text="Tipo de desdobramento"
)

fig.show()


### V2 — Small multiples por tipo de desdobramento

**Objetivo:** reduzir a poluição visual quando há muitos tipos de desdobramentos.

**Quando usar no Streamlit:**  
Use se o gráfico de linha com várias cores ficar confuso. Cada tipo fica separado em um pequeno gráfico, facilitando a leitura individual das tendências.

In [12]:
fig = px.line(
    df_linha_top,
    x="data",
    y="valor",
    facet_col="indicador_completo",
    facet_col_wrap=2,
    markers=True,
    title="Evolução individual dos tipos de desdobramentos técnicos",
    labels={
        "data": "Período",
        "valor": "Quantidade",
        "indicador_completo": "Tipo de desdobramento técnico"
    },
    hover_data={
        "ano": True,
        "mes": True,
        "valor": True,
        "indicador_completo": True
    }
)

fig.update_layout(
    height=900,
    showlegend=False
)

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig.show()


### V3 — Variação percentual por tipo de desdobramento

**Objetivo:** identificar quais tipos mais cresceram ou diminuíram entre o primeiro e o último ano selecionado.

**Quando usar no Streamlit:**  
Use como visualização complementar ao gráfico de linha. Ela ajuda a resumir quais desdobramentos tiveram maior crescimento ou queda no período.

In [13]:
df_anual_desdobramentos = (
    df_desdobramentos_filtrado
    .groupby(["ano", "indicador_completo"], as_index=False)
    .agg(total=("valor", "sum"))
)

primeiro_ano = min(anos_selecionados_q2)
ultimo_ano = max(anos_selecionados_q2)

df_inicio = df_anual_desdobramentos[df_anual_desdobramentos["ano"] == primeiro_ano][
    ["indicador_completo", "total"]
].rename(columns={"total": "total_inicio"})

df_fim = df_anual_desdobramentos[df_anual_desdobramentos["ano"] == ultimo_ano][
    ["indicador_completo", "total"]
].rename(columns={"total": "total_fim"})

df_variacao = df_inicio.merge(df_fim, on="indicador_completo", how="outer").fillna(0)

df_variacao["variacao_absoluta"] = df_variacao["total_fim"] - df_variacao["total_inicio"]

df_variacao["variacao_percentual"] = np.where(
    df_variacao["total_inicio"] > 0,
    (df_variacao["variacao_absoluta"] / df_variacao["total_inicio"]) * 100,
    np.nan
)

df_variacao_plot = (
    df_variacao
    .sort_values("variacao_absoluta", ascending=False)
    .head(15)
)

fig = px.bar(
    df_variacao_plot,
    x="variacao_absoluta",
    y="indicador_completo",
    orientation="h",
    title=f"Variação dos tipos de desdobramentos técnicos entre {primeiro_ano} e {ultimo_ano}",
    labels={
        "variacao_absoluta": "Variação absoluta",
        "indicador_completo": "Tipo de desdobramento técnico"
    },
    hover_data={
        "total_inicio": True,
        "total_fim": True,
        "variacao_percentual": ":.1f"
    }
)

fig.update_layout(
    height=650,
    yaxis={"categoryorder": "total ascending"}
)

fig.show()


# 7. Pergunta 3 — Padrões sazonais e períodos críticos

**Pergunta de negócio:**  
Existem padrões sazonais ou períodos críticos nos indicadores de entrada, saída, evasão e desligamento das crianças e adolescentes?

Para esta pergunta, serão analisados os indicadores relacionados a movimentações institucionais, como entrada, saída, evasão e desligamento. O objetivo é identificar meses com maior concentração desses eventos e verificar possíveis padrões sazonais ao longo dos anos.

#### Filtro dos indicadores de entrada, saída, evasão e desligamento

Nesta etapa, são selecionados apenas os indicadores relacionados à movimentação das crianças e adolescentes na instituição.

In [14]:
# Seleção dos indicadores relacionados à Pergunta 3.
# O filtro usa palavras-chave para localizar entrada, saída, evasão e desligamento
# nas colunas textuais da base.

import re

palavras_chave_q3 = [
    "entrada",
    "entradas",
    "saída",
    "saida",
    "saídas",
    "saidas",
    "evasão",
    "evasao",
    "desligamento",
    "desligamentos"
]

colunas_texto_q3 = ["categoria", "subcategoria", "indicador", "indicador_completo"]

mascara_q3 = pd.Series(False, index=df.index)

padrao_q3 = "|".join(re.escape(palavra) for palavra in palavras_chave_q3)

for coluna in colunas_texto_q3:
    if coluna in df.columns:
        texto_coluna = (
            df[coluna]
            .fillna("")
            .astype(str)
            .str.lower()
        )

        mascara_q3 = mascara_q3 | texto_coluna.str.contains(
            padrao_q3,
            regex=True,
            na=False
        )

df_movimentacoes = df.loc[mascara_q3].copy()

print("Quantidade de registros encontrados:", len(df_movimentacoes))

display(
    df_movimentacoes[
        ["ano", "mes", "categoria", "subcategoria", "indicador", "indicador_completo", "valor"]
    ].head()
)

print("Indicadores encontrados para a Pergunta 3:")
display(sorted(df_movimentacoes["indicador_completo"].dropna().unique()))

Quantidade de registros encontrados: 120


,ano,mes,categoria,subcategoria,indicador,indicador_completo,valor
20,2021,JAN,NaN,NaN,Desligamentos,Desligamentos,0.0
23,2021,JAN,NaN,NaN,Evasão,Evasão,2.0
53,2021,FEV,NaN,NaN,Desligamentos,Desligamentos,1.0
56,2021,FEV,NaN,NaN,Evasão,Evasão,1.0
86,2021,MAR,NaN,NaN,Desligamentos,Desligamentos,0.0


Indicadores encontrados para a Pergunta 3:


['Desligamentos', 'Evasão']

#### Filtro de período mensal para teste

No Streamlit, esse filtro pode virar um seletor de intervalo de datas ou um filtro por mês e ano.  
No notebook, ele fica como variáveis simples para facilitar os testes.

In [15]:
# Filtros para testar no notebook.
# Depois, no Streamlit, isso pode virar um filtro de período mensal.

if len(df_movimentacoes) > 0:
    data_min_q3 = df_movimentacoes["data"].min()
    data_max_q3 = df_movimentacoes["data"].max()

    periodo_inicio_q3 = data_min_q3
    periodo_fim_q3 = data_max_q3

    df_movimentacoes_filtrado = df_movimentacoes[
        (df_movimentacoes["data"] >= periodo_inicio_q3) &
        (df_movimentacoes["data"] <= periodo_fim_q3)
    ].copy()

    print("Período disponível:", data_min_q3.strftime("%m/%Y"), "até", data_max_q3.strftime("%m/%Y"))
    print("Período selecionado:", periodo_inicio_q3.strftime("%m/%Y"), "até", periodo_fim_q3.strftime("%m/%Y"))
    print("Quantidade após filtro:", len(df_movimentacoes_filtrado))
else:
    df_movimentacoes_filtrado = df_movimentacoes.copy()
    print("Nenhum registro encontrado para os indicadores da Pergunta 3.")

Período disponível: 01/2021 até 12/2025
Período selecionado: 01/2021 até 12/2025
Quantidade após filtro: 120


### V1 — Gráfico de barras por mês para indicadores de entrada, saída, evasão e desligamento

**Objetivo:** identificar padrões sazonais e períodos críticos nos indicadores de movimentação das crianças e adolescentes.

**Quando usar no Streamlit:**  
Esta é a visualização principal para a terceira pergunta. O gráfico de barras facilita a comparação entre meses e ajuda a identificar picos, quedas e padrões sazonais. Com um filtro de período mensal, é possível selecionar um intervalo específico de análise, tornando mais clara a identificação de períodos críticos.

In [16]:
if len(df_movimentacoes_filtrado) > 0:
    df_barras_mensal = (
        df_movimentacoes_filtrado
        .groupby(["data", "ano", "mes", "mes_numero", "indicador_completo"], as_index=False)
        .agg(valor=("valor", "sum"))
        .sort_values("data")
    )

    # Para evitar excesso de barras, mantém os principais indicadores pelo volume total.
    top_indicadores_q3 = (
        df_barras_mensal
        .groupby("indicador_completo", as_index=False)
        .agg(total=("valor", "sum"))
        .sort_values("total", ascending=False)
        .head(8)["indicador_completo"]
    )

    df_barras_top = df_barras_mensal[
        df_barras_mensal["indicador_completo"].isin(top_indicadores_q3)
    ].copy()

    df_barras_top["periodo"] = df_barras_top["data"].dt.strftime("%m/%Y")

    fig = px.bar(
        df_barras_top,
        x="periodo",
        y="valor",
        color="indicador_completo",
        barmode="group",
        title="Indicadores de entrada, saída, evasão e desligamento por mês",
        labels={
            "periodo": "Período mensal",
            "valor": "Quantidade",
            "indicador_completo": "Indicador"
        },
        hover_data={
            "ano": True,
            "mes": True,
            "valor": True,
            "indicador_completo": True
        }
    )

    fig.update_layout(
        height=650,
        xaxis_tickangle=-45,
        legend_title_text="Indicador"
    )

    fig.show()
else:
    print("Não há dados suficientes para gerar o gráfico da Pergunta 3.")

### V2 — Heatmap mensal dos períodos críticos

**Objetivo:** destacar rapidamente os meses e anos com maior concentração de movimentações.

**Quando usar no Streamlit:**  
Use como visualização complementar ao gráfico de barras. O heatmap facilita a identificação visual dos períodos mais críticos, pois concentra a análise no volume total mensal dos indicadores selecionados.

In [17]:
if len(df_movimentacoes_filtrado) > 0:
    df_heatmap_q3 = (
        df_movimentacoes_filtrado
        .groupby(["ano", "mes_numero", "mes"], as_index=False)
        .agg(total=("valor", "sum"))
        .sort_values(["ano", "mes_numero"])
    )

    matriz_q3 = df_heatmap_q3.pivot_table(
        index="mes",
        columns="ano",
        values="total",
        aggfunc="sum",
        fill_value=0
    )

    ordem_meses = (
        df_heatmap_q3[["mes", "mes_numero"]]
        .drop_duplicates()
        .sort_values("mes_numero")["mes"]
        .tolist()
    )

    matriz_q3 = matriz_q3.reindex(ordem_meses)

    fig = px.imshow(
        matriz_q3,
        text_auto=True,
        aspect="auto",
        title="Mapa de calor dos períodos críticos de movimentação",
        labels={
            "x": "Ano",
            "y": "Mês",
            "color": "Quantidade"
        }
    )

    fig.update_layout(height=600)

    fig.show()
else:
    print("Não há dados suficientes para gerar o heatmap da Pergunta 3.")

# 8. Pergunta 4 — Estabilidade da permanência na instituição

**Pergunta de negócio:**  
A permanência das crianças e adolescentes na instituição está se tornando mais estável ou mais instável ao longo dos anos?

Para esta pergunta, será utilizado um diagrama de fluxo para observar movimentações ao longo dos anos. Como a base está agregada por indicadores, o fluxo representa volumes consolidados de entradas, permanências e saídas, e não trajetórias individuais de cada criança ou adolescente.

#### Preparação dos fluxos

Nesta etapa, os indicadores são classificados em três grupos principais: entradas, permanências e saídas. Essa classificação permite construir um diagrama de fluxo para observar a estabilidade ou instabilidade da permanência ao longo dos anos.

In [18]:
# Classificação dos indicadores em tipos de fluxo.
# Essa função pode ser ajustada caso novos nomes de indicadores apareçam na base.

def classificar_fluxo(texto):
    texto = str(texto).lower()

    if any(palavra in texto for palavra in ["entrada", "entradas", "ingresso", "ingressos"]):
        return "Entrada"

    if any(palavra in texto for palavra in ["saída", "saida", "saídas", "saidas", "evasão", "evasao", "desligamento", "desligamentos"]):
        return "Saída"

    if any(palavra in texto for palavra in ["permanência", "permanencia", "permanece", "permanecem", "ativos", "ativo"]):
        return "Permanência"

    return None


df_fluxo = df.copy()

df_fluxo["tipo_fluxo"] = df_fluxo["indicador_completo"].apply(classificar_fluxo)

# Caso o nome mais claro esteja em outra coluna, tenta complementar a classificação.
for coluna in ["indicador", "subcategoria", "categoria"]:
    if coluna in df_fluxo.columns:
        mascara_sem_classificacao = df_fluxo["tipo_fluxo"].isna()
        df_fluxo.loc[mascara_sem_classificacao, "tipo_fluxo"] = (
            df_fluxo.loc[mascara_sem_classificacao, coluna]
            .apply(classificar_fluxo)
        )

df_fluxo = df_fluxo[df_fluxo["tipo_fluxo"].notna()].copy()

print("Quantidade de registros classificados para o fluxo:", len(df_fluxo))

display(
    df_fluxo[
        ["ano", "mes", "tipo_fluxo", "categoria", "subcategoria", "indicador", "indicador_completo", "valor"]
    ].head()
)

print("Tipos de fluxo encontrados:")
display(df_fluxo["tipo_fluxo"].value_counts())

Quantidade de registros classificados para o fluxo: 228


,ano,mes,tipo_fluxo,categoria,subcategoria,indicador,indicador_completo,valor
6,2021,JAN,Permanência,EDUCAÇÃO,SCFV,"Outros: Reforço escolar, psicopedagoga, trabal...","SCFV - Outros: Reforço escolar, psicopedagoga,...",0.0
20,2021,JAN,Saída,NaN,NaN,Desligamentos,Desligamentos,0.0
23,2021,JAN,Saída,NaN,NaN,Evasão,Evasão,2.0
28,2021,JAN,Entrada,NaN,NaN,Novos ingressos,Novos ingressos,0.0
39,2021,FEV,Permanência,EDUCAÇÃO,SCFV,"Outros: Reforço escolar, psicopedagoga, trabal...","SCFV - Outros: Reforço escolar, psicopedagoga,...",0.0


Tipos de fluxo encontrados:


tipo_fluxo
Saída          120
Entrada         60
Permanência     48
Name: count, dtype: int64

#### Filtro de ano para teste

No Streamlit, o isolamento de fluxo pode ser feito por seleção de ano ou por seleção de nó no próprio gráfico.  
No notebook, deixamos um seletor simples por variável.

In [19]:
if len(df_fluxo) > 0:
    anos_disponiveis_q4 = sorted(df_fluxo["ano"].dropna().unique())

    # Para visualizar todos os anos, mantém todos selecionados.
    # Para testar um ano específico, substitua por exemplo: anos_selecionados_q4 = [2024]
    anos_selecionados_q4 = anos_disponiveis_q4

    df_fluxo_filtrado = df_fluxo[
        df_fluxo["ano"].isin(anos_selecionados_q4)
    ].copy()

    print("Anos disponíveis:", anos_disponiveis_q4)
    print("Anos selecionados:", anos_selecionados_q4)
    print("Quantidade após filtro:", len(df_fluxo_filtrado))
else:
    df_fluxo_filtrado = df_fluxo.copy()
    anos_disponiveis_q4 = []
    anos_selecionados_q4 = []
    print("Nenhum registro encontrado para construir o fluxo.")

Anos disponíveis: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Anos selecionados: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Quantidade após filtro: 228


### V1 — Diagrama de fluxo das movimentações institucionais

**Objetivo:** visualizar as movimentações das crianças e adolescentes ao longo do tempo, evidenciando entradas, permanências e saídas.

**Quando usar no Streamlit:**  
Esta é a visualização principal para a quarta pergunta. O diagrama de fluxo permite observar como os volumes se distribuem entre entradas, permanências e saídas ao longo dos anos. Com o isolamento de fluxo por seleção de nó, é possível analisar um tipo específico de movimentação e verificar se a permanência aparenta estar mais estável ou mais instável em um ano específico ou no período completo.

In [20]:
if len(df_fluxo_filtrado) > 0:
    df_sankey = (
        df_fluxo_filtrado
        .groupby(["ano", "tipo_fluxo"], as_index=False)
        .agg(valor=("valor", "sum"))
        .sort_values(["ano", "tipo_fluxo"])
    )

    # Nós do Sankey:
    # Ano -> Tipo de fluxo
    anos_nodes = [f"Ano {int(ano)}" for ano in sorted(df_sankey["ano"].dropna().unique())]
    fluxo_nodes = ["Entrada", "Permanência", "Saída"]

    labels = anos_nodes + fluxo_nodes
    label_to_index = {label: i for i, label in enumerate(labels)}

    sources = []
    targets = []
    values = []

    for _, row in df_sankey.iterrows():
        ano_label = f"Ano {int(row['ano'])}"
        fluxo_label = row["tipo_fluxo"]

        sources.append(label_to_index[ano_label])
        targets.append(label_to_index[fluxo_label])
        values.append(row["valor"])

    fig = go.Figure(
        data=[
            go.Sankey(
                node=dict(
                    pad=18,
                    thickness=18,
                    line=dict(width=0.5),
                    label=labels
                ),
                link=dict(
                    source=sources,
                    target=targets,
                    value=values
                )
            )
        ]
    )

    fig.update_layout(
        title_text="Fluxo de entradas, permanências e saídas por ano",
        height=650
    )

    fig.show()
else:
    print("Não há dados suficientes para gerar o diagrama de fluxo da Pergunta 4.")

### V2 — Índice simples de estabilidade por ano

**Objetivo:** complementar o diagrama de fluxo com uma medida resumida de estabilidade.

**Quando usar no Streamlit:**  
Use como apoio ao diagrama de fluxo. A ideia é comparar permanências com entradas e saídas. Quando o peso das permanências é maior em relação às movimentações de entrada e saída, o cenário pode indicar maior estabilidade. Quando entradas e saídas ganham mais peso, o cenário pode sugerir maior instabilidade.

In [21]:
if len(df_fluxo_filtrado) > 0:
    df_estabilidade = (
        df_fluxo_filtrado
        .groupby(["ano", "tipo_fluxo"], as_index=False)
        .agg(valor=("valor", "sum"))
        .pivot_table(
            index="ano",
            columns="tipo_fluxo",
            values="valor",
            aggfunc="sum",
            fill_value=0
        )
        .reset_index()
    )

    for coluna in ["Entrada", "Permanência", "Saída"]:
        if coluna not in df_estabilidade.columns:
            df_estabilidade[coluna] = 0

    df_estabilidade["movimentacoes"] = df_estabilidade["Entrada"] + df_estabilidade["Saída"]

    df_estabilidade["indice_estabilidade"] = np.where(
        (df_estabilidade["Permanência"] + df_estabilidade["movimentacoes"]) > 0,
        df_estabilidade["Permanência"] / (df_estabilidade["Permanência"] + df_estabilidade["movimentacoes"]),
        0
    )

    display(df_estabilidade)

    fig = px.line(
        df_estabilidade,
        x="ano",
        y="indice_estabilidade",
        markers=True,
        title="Índice simples de estabilidade da permanência por ano",
        labels={
            "ano": "Ano",
            "indice_estabilidade": "Índice de estabilidade"
        },
        hover_data={
            "Entrada": True,
            "Permanência": True,
            "Saída": True,
            "movimentacoes": True
        }
    )

    fig.update_layout(height=500)

    fig.show()
else:
    print("Não há dados suficientes para calcular o índice de estabilidade.")

tipo_fluxo,ano,Entrada,Permanência,Saída,movimentacoes,indice_estabilidade
0,2021,11.0,0.0,19.0,30.0,0.000000
1,2022,9.0,0.0,28.0,37.0,0.000000
2,2023,19.0,7.0,42.0,61.0,0.102941
3,2024,7.0,0.0,61.0,68.0,0.000000
4,2025,12.0,90.0,14.0,26.0,0.775862


## 5.5 — Proporcionalidade entre acolhidos e indicadores de saúde e educação

**Pergunta de negócio:**  
A evolução dos indicadores de saúde mental, saúde clínica e educação acompanha proporcionalmente a variação no número de acolhidos?

Para esta pergunta, será utilizado um gráfico combinado de barras e linhas. As barras representam o número de acolhidos, enquanto as linhas mostram a evolução dos indicadores selecionados. Essa combinação permite comparar se os indicadores acompanham, aumentam ou diminuem proporcionalmente em relação ao volume de acolhidos.

#### Preparação dos dados de acolhidos, saúde mental, saúde clínica e educação

Nesta etapa, os registros são classificados em quatro grupos de análise: número de acolhidos, saúde mental, saúde clínica e educação. A classificação é feita a partir dos textos das colunas de categoria, subcategoria, indicador e indicador completo.

In [22]:
# Função auxiliar para buscar palavras-chave nas principais colunas de texto.

def contem_palavra_chave_linha(linha, palavras_chave):
    texto = " ".join([
        str(linha.get("categoria", "")),
        str(linha.get("subcategoria", "")),
        str(linha.get("indicador", "")),
        str(linha.get("indicador_completo", ""))
    ]).lower()

    return any(palavra in texto for palavra in palavras_chave)


def classificar_grupo_q5(linha):
    if contem_palavra_chave_linha(linha, ["acolhido", "acolhidos", "acolhida", "acolhidas", "acolhimento"]):
        return "Número de acolhidos"

    if contem_palavra_chave_linha(linha, ["saúde mental", "saude mental", "psicol", "psiquiatr", "terapia"]):
        return "Saúde mental"

    if contem_palavra_chave_linha(linha, ["saúde clínica", "saude clinica", "saúde clinica", "saude clínica", "clínica", "clinica", "médic", "medic", "consulta", "enferm"]):
        return "Saúde clínica"

    if contem_palavra_chave_linha(linha, ["educação", "educacao", "escola", "escolar", "pedag", "ensino"]):
        return "Educação"

    return None


df_q5 = df.copy()
df_q5["grupo_analise"] = df_q5.apply(classificar_grupo_q5, axis=1)
df_q5 = df_q5[df_q5["grupo_analise"].notna()].copy()

print("Quantidade de registros encontrados para a Pergunta 5:", len(df_q5))
print("Grupos encontrados:")
display(df_q5["grupo_analise"].value_counts())

display(
    df_q5[["ano", "mes", "categoria", "subcategoria", "indicador", "indicador_completo", "grupo_analise", "valor"]]
    .head(20)
)

Quantidade de registros encontrados para a Pergunta 5: 612
Grupos encontrados:


grupo_analise
Educação         496
Saúde clínica     58
Saúde mental      58
Name: count, dtype: int64

,ano,mes,categoria,subcategoria,indicador,indicador_completo,grupo_analise,valor
0,2021,JAN,EDUCAÇÃO,Ensino EJA,aguardando vaga,Ensino EJA - aguardando vaga,Educação,0.0
1,2021,JAN,EDUCAÇÃO,Ensino EJA,matriculados,Ensino EJA - matriculados,Educação,0.0
2,2021,JAN,EDUCAÇÃO,Ensino infantil,aguardando vaga,Ensino infantil - aguardando vaga,Educação,3.0
3,2021,JAN,EDUCAÇÃO,Ensino infantil,matriculados,Ensino infantil - matriculados,Educação,2.0
4,2021,JAN,EDUCAÇÃO,Ensino regular,aguardando vaga,Ensino regular - aguardando vaga,Educação,3.0
5,2021,JAN,EDUCAÇÃO,Ensino regular,matriculados,Ensino regular - matriculados,Educação,12.0
6,2021,JAN,EDUCAÇÃO,SCFV,"Outros: Reforço escolar, psicopedagoga, trabal...","SCFV - Outros: Reforço escolar, psicopedagoga,...",Educação,0.0
7,2021,JAN,EDUCAÇÃO,SCFV,aguardando vaga,SCFV - aguardando vaga,Educação,0.0
8,2021,JAN,EDUCAÇÃO,SCFV,matriculados,SCFV - matriculados,Educação,0.0
14,2021,JAN,SAÚDE,NaN,Saúde clínica,Saúde clínica,Saúde clínica,11.0


#### Filtro de anos para teste

No Streamlit, esse filtro poderá ser transformado em um seletor de anos. No notebook, ele fica como uma variável para facilitar os testes.

In [23]:
if len(df_q5) > 0:
    anos_disponiveis_q5 = sorted(df_q5["ano"].dropna().unique())
    anos_selecionados_q5 = anos_disponiveis_q5

    df_q5_filtrado = df_q5[df_q5["ano"].isin(anos_selecionados_q5)].copy()

    print("Anos disponíveis:", anos_disponiveis_q5)
    print("Anos selecionados:", anos_selecionados_q5)
    print("Quantidade após filtro:", len(df_q5_filtrado))
else:
    df_q5_filtrado = df_q5.copy()
    print("Nenhum registro encontrado para a Pergunta 5. Verifique as palavras-chave usadas na classificação.")

Anos disponíveis: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Anos selecionados: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Quantidade após filtro: 612


### V1 — Gráfico combinado de barras e linhas

**Objetivo:** comparar a evolução do número de acolhidos com os indicadores de saúde mental, saúde clínica e educação.

**Quando usar no Streamlit:**  
Esta é a visualização principal para a quinta pergunta. As barras mostram o número de acolhidos e as linhas mostram os indicadores analisados, permitindo observar se as demandas de saúde e educação crescem ou diminuem de forma proporcional ao volume de crianças e adolescentes acolhidos.

In [24]:
from plotly.subplots import make_subplots

if len(df_q5_filtrado) > 0:
    df_q5_anual = (
        df_q5_filtrado
        .groupby(["ano", "grupo_analise"], as_index=False)
        .agg(valor=("valor", "sum"))
        .sort_values(["ano", "grupo_analise"])
    )

    df_q5_pivot = df_q5_anual.pivot_table(
        index="ano",
        columns="grupo_analise",
        values="valor",
        aggfunc="sum",
        fill_value=0
    ).reset_index()

    fig = make_subplots(specs=[[{"secondary_y": True}]])

    if "Número de acolhidos" in df_q5_pivot.columns:
        fig.add_trace(
            go.Bar(
                x=df_q5_pivot["ano"],
                y=df_q5_pivot["Número de acolhidos"],
                name="Número de acolhidos"
            ),
            secondary_y=False
        )

    for grupo in ["Saúde mental", "Saúde clínica", "Educação"]:
        if grupo in df_q5_pivot.columns:
            fig.add_trace(
                go.Scatter(
                    x=df_q5_pivot["ano"],
                    y=df_q5_pivot[grupo],
                    mode="lines+markers",
                    name=grupo
                ),
                secondary_y=True
            )

    fig.update_layout(
        title="Número de acolhidos x indicadores de saúde e educação por ano",
        xaxis_title="Ano",
        barmode="group",
        hovermode="x unified",
        height=600
    )

    fig.update_yaxes(title_text="Número de acolhidos", secondary_y=False)
    fig.update_yaxes(title_text="Indicadores de saúde e educação", secondary_y=True)

    fig.show()
else:
    print("Não há dados suficientes para gerar o gráfico combinado da Pergunta 5.")

### V2 — Evolução proporcional com índice base 100

**Objetivo:** comparar a variação proporcional dos grupos ao longo dos anos, usando o primeiro ano disponível como base 100.

**Quando usar no Streamlit:**  
Use como complemento ao gráfico combinado. Essa visualização facilita a comparação proporcional, pois todos os grupos começam na mesma base. Assim, fica mais claro perceber se saúde mental, saúde clínica e educação crescem mais rápido, mais devagar ou em ritmo semelhante ao número de acolhidos.

In [25]:
if len(df_q5_filtrado) > 0:
    df_q5_anual = (
        df_q5_filtrado
        .groupby(["ano", "grupo_analise"], as_index=False)
        .agg(valor=("valor", "sum"))
        .sort_values(["grupo_analise", "ano"])
    )

    # Calcula o índice base 100 sem usar groupby.apply,
    # evitando que a coluna grupo_analise desapareça no resultado.
    df_q5_indice = df_q5_anual.copy()

    df_q5_indice["valor_base"] = (
        df_q5_indice
        .groupby("grupo_analise")["valor"]
        .transform("first")
    )

    df_q5_indice["indice_base_100"] = np.where(
        df_q5_indice["valor_base"] != 0,
        (df_q5_indice["valor"] / df_q5_indice["valor_base"]) * 100,
        np.nan
    )

    fig = px.line(
        df_q5_indice,
        x="ano",
        y="indice_base_100",
        color="grupo_analise",
        markers=True,
        title="Evolução proporcional dos acolhidos e dos indicadores — índice base 100",
        labels={
            "ano": "Ano",
            "indice_base_100": "Índice base 100",
            "grupo_analise": "Grupo de análise"
        },
        hover_data={"valor": True, "grupo_analise": True}
    )

    fig.update_layout(height=600, hovermode="x unified")
    fig.show()
else:
    print("Não há dados suficientes para gerar o índice base 100 da Pergunta 5.")


## 5.6 — Distribuição e alcance das atividades realizadas em 2025

**Pergunta de negócio:**  
As atividades culturais, capacitações e ações institucionais realizadas em 2025 estão distribuídas de forma equilibrada ao longo do ano e quais apresentam maior alcance em participantes, locais e carga horária?

Para esta pergunta, será utilizado um treemap. A área dos retângulos pode representar diferentes métricas, como número de participantes, carga horária total ou quantidade de eventos realizados. No Streamlit, essa escolha pode ser feita por botões de rádio na barra lateral.

#### Preparação dos dados de atividades de 2025

Nesta etapa, são filtrados os registros de 2025 relacionados a atividades culturais, capacitações e ações institucionais. Também é criada uma classificação da métrica analisada, separando participantes, carga horária e quantidade de eventos.

In [26]:
def classificar_tipo_atividade_q6(linha):
    texto = " ".join([
        str(linha.get("categoria", "")),
        str(linha.get("subcategoria", "")),
        str(linha.get("indicador", "")),
        str(linha.get("indicador_completo", ""))
    ]).lower()

    if any(palavra in texto for palavra in ["cultura", "cultural", "oficina", "passeio", "evento cultural"]):
        return "Atividades culturais"

    if any(palavra in texto for palavra in ["capacitação", "capacitacao", "formação", "formacao", "treinamento", "curso"]):
        return "Capacitações"

    if any(palavra in texto for palavra in ["ação institucional", "acao institucional", "ações institucionais", "acoes institucionais", "institucional"]):
        return "Ações institucionais"

    return None


def classificar_metrica_q6(linha):
    texto = " ".join([
        str(linha.get("categoria", "")),
        str(linha.get("subcategoria", "")),
        str(linha.get("indicador", "")),
        str(linha.get("indicador_completo", ""))
    ]).lower()

    if any(palavra in texto for palavra in ["participante", "participantes", "público", "publico", "pessoas"]):
        return "Número de Participantes"

    if any(palavra in texto for palavra in ["carga horária", "carga horaria", "horas", "hora"]):
        return "Carga Horária Total"

    if any(palavra in texto for palavra in ["evento", "eventos", "atividade", "atividades", "ações", "acoes", "ação", "acao", "realizados", "realizadas"]):
        return "Quantidade de Eventos Realizados"

    return None


df_q6 = df[df["ano"] == 2025].copy()
df_q6["tipo_atividade"] = df_q6.apply(classificar_tipo_atividade_q6, axis=1)
df_q6["metrica_q6"] = df_q6.apply(classificar_metrica_q6, axis=1)

df_q6 = df_q6[
    df_q6["tipo_atividade"].notna() &
    df_q6["metrica_q6"].notna()
].copy()

# Nome da atividade usado no treemap.
# Caso a base tenha uma coluna mais específica no futuro, ela pode substituir indicador_completo aqui.
df_q6["atividade"] = df_q6["indicador_completo"].fillna(df_q6["indicador"]).fillna("Atividade não informada")

print("Quantidade de registros encontrados para a Pergunta 6:", len(df_q6))
print("Tipos de atividades encontrados:")
display(df_q6["tipo_atividade"].value_counts())
print("Métricas encontradas:")
display(df_q6["metrica_q6"].value_counts())

display(
    df_q6[["ano", "mes", "tipo_atividade", "metrica_q6", "atividade", "valor"]]
    .head(20)
)

Quantidade de registros encontrados para a Pergunta 6: 24
Tipos de atividades encontrados:


tipo_atividade
Capacitações    24
Name: count, dtype: int64

Métricas encontradas:


metrica_q6
Quantidade de Eventos Realizados    24
Name: count, dtype: int64

,ano,mes,tipo_atividade,metrica_q6,atividade,valor
1483,2025,JAN,Capacitações,Quantidade de Eventos Realizados,Encaminhados para curso profissionalizante,4.0
1485,2025,JAN,Capacitações,Quantidade de Eventos Realizados,Inseridos em curso profissionalizante,0.0
1516,2025,FEV,Capacitações,Quantidade de Eventos Realizados,Encaminhados para curso profissionalizante,1.0
1518,2025,FEV,Capacitações,Quantidade de Eventos Realizados,Inseridos em curso profissionalizante,4.0
1549,2025,MAR,Capacitações,Quantidade de Eventos Realizados,Encaminhados para curso profissionalizante,1.0
1551,2025,MAR,Capacitações,Quantidade de Eventos Realizados,Inseridos em curso profissionalizante,4.0
1582,2025,ABR,Capacitações,Quantidade de Eventos Realizados,Encaminhados para curso profissionalizante,0.0
1584,2025,ABR,Capacitações,Quantidade de Eventos Realizados,Inseridos em curso profissionalizante,0.0
1615,2025,MAI,Capacitações,Quantidade de Eventos Realizados,Encaminhados para curso profissionalizante,1.0
1617,2025,MAI,Capacitações,Quantidade de Eventos Realizados,Inseridos em curso profissionalizante,0.0


#### Seleção da métrica de proporção para teste

No Streamlit, esta seleção será feita com botões de rádio na barra lateral. No notebook, basta alterar a variável `metrica_selecionada_q6`.

In [ ]:
metricas_disponiveis_q6 = [
    "Número de Participantes",
    "Carga Horária Total",
    "Quantidade de Eventos Realizados"
]

# Altere esta variável para testar outra métrica no treemap.
# No Streamlit, esta escolha vira radio button na barra lateral.
metrica_selecionada_q6 = "Quantidade de Eventos Realizados"

if len(df_q6) > 0:
    metricas_encontradas_q6 = sorted(df_q6["metrica_q6"].dropna().unique())

    if metrica_selecionada_q6 not in metricas_encontradas_q6:
        metrica_selecionada_q6 = metricas_encontradas_q6[0]

    df_q6_filtrado = df_q6[
        df_q6["metrica_q6"] == metrica_selecionada_q6
    ].copy()

    print("Métricas disponíveis na base:", metricas_encontradas_q6)
    print("Métrica selecionada:", metrica_selecionada_q6)
    print("Quantidade de registros para a métrica selecionada:", len(df_q6_filtrado))
else:
    df_q6_filtrado = df_q6.copy()
    print("Nenhum registro encontrado para a Pergunta 6. Verifique as palavras-chave usadas na classificação.")


Métricas disponíveis na base: ['Quantidade de Eventos Realizados']
Métrica selecionada: Quantidade de Eventos Realizados
Quantidade de registros para a métrica selecionada: 24


### V1 — Treemap das atividades realizadas em 2025

**Objetivo:** visualizar a proporção e o peso das atividades culturais, capacitações e ações institucionais realizadas em 2025.

**Quando usar no Streamlit:**  
Esta é a visualização principal para a sexta pergunta. O treemap permite identificar quais atividades tiveram maior alcance ou impacto dentro do conjunto analisado. No Streamlit, a métrica que define o tamanho dos retângulos pode ser alterada por botões de rádio, permitindo comparar número de participantes, carga horária total ou quantidade de eventos realizados.

In [ ]:
if len(df_q6_filtrado) > 0:
    df_treemap_q6 = (
        df_q6_filtrado
        .groupby(["tipo_atividade", "atividade"], as_index=False)
        .agg(valor=("valor", "sum"))
        .sort_values("valor", ascending=False)
    )
    df_treemap_q6 = df_treemap_q6[df_treemap_q6["valor"] > 0]

    if len(df_treemap_q6) > 0:
        fig = px.treemap(
            df_treemap_q6,
            path=["tipo_atividade", "atividade"],
            values="valor",
            title=f"Treemap das atividades de 2025 por {metrica_selecionada_q6}",
            hover_data={"valor": True}
        )

        fig.update_traces(
            hovertemplate=(
                "<b>%{label}</b><br>" +
                f"<b>Métrica:</b> {metrica_selecionada_q6}<br>" +
                "<b>Valor:</b> %{value}<extra></extra>"
            )
        )

        fig.update_layout(height=700)
        fig.show()
    else:
        print("Existem registros para a Pergunta 6, mas todos os valores da métrica selecionada são zero.")
else:
    print("Nenhum registro encontrado para a métrica selecionada.")


### V2 — Distribuição mensal das atividades em 2025

**Objetivo:** verificar se as atividades culturais, capacitações e ações institucionais estão distribuídas de forma equilibrada ao longo do ano.

**Quando usar no Streamlit:**  
Use como complemento ao treemap. Enquanto o treemap destaca quais atividades tiveram maior peso, o gráfico de barras mensal ajuda a identificar concentração de ações em determinados meses e possíveis períodos com maior intensidade de atividades.

In [ ]:
if len(df_q6_filtrado) > 0:
    df_barras_q6 = (
        df_q6_filtrado
        .groupby(["mes_numero", "mes", "tipo_atividade"], as_index=False)
        .agg(valor=("valor", "sum"))
        .sort_values("mes_numero")
    )

    df_barras_q6 = df_barras_q6[df_barras_q6["valor"] > 0]

    if len(df_barras_q6) > 0:
        fig = px.bar(
            df_barras_q6,
            x="mes",
            y="valor",
            color="tipo_atividade",
            barmode="group",
            title=f"Distribuição mensal das atividades de 2025 por {metrica_selecionada_q6}",
            labels={
                "mes": "Mês",
                "valor": metrica_selecionada_q6,
                "tipo_atividade": "Tipo de atividade"
            },
            hover_data={"mes_numero": False, "tipo_atividade": True, "valor": True}
        )

        fig.update_layout(height=600, xaxis_title="Mês", yaxis_title=metrica_selecionada_q6)
        fig.show()
    else:
        print("Existem registros para a Pergunta 6, mas todos os valores da métrica selecionada são zero.")
else:
    print("Nenhum registro encontrado para a métrica selecionada.")
